# ⚡ Kaggle 24/7 OSINT Agent + HOT RELOAD INSTANTANÉ (2 secondes)
Optimisation ultime : Le modèle LLM 7 Go reste chargé en mémoire RAM/GPU. Les modifications d'interface et du backend FastAPI sont appliquées en 2 secondes via Git Pull sans recharger le modèle.

In [ ]:
# 1. Redirection TMPDIR & Initialisation des dossiers
import os

os.environ['TMPDIR'] = '/kaggle/working/tmp'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/tmp/pip'
os.makedirs('/kaggle/working/tmp', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)
print('🟢 Dossiers temporaires et modèles prêts !')

In [ ]:
# 2. Clonage ou Mise à jour ultra-rapide par Git Pull (1 seconde)
import os, subprocess, time

repo_dir = '/kaggle/working/projet_osint'
clone_url = 'https://github.com/whbky6vqjb-coder/osint.git'

if os.path.exists(repo_dir):
    print('⚡ Dépôt déjà présent : Exécution d\'un Git Pull ultra-rapide (1 sec)...')
    res = subprocess.run(['git', '-C', repo_dir, 'pull', 'origin', 'main'])
    if res.returncode == 0:
        print('🟢 Code source mis à jour instantanément !')
else:
    print('📥 Premier clonage du dépôt Git...')
    subprocess.run(['git', 'clone', clone_url, repo_dir])
    print('🟢 Dépôt Git cloné !')

!pip install --no-cache-dir --prefer-binary huggingface_hub "llama-cpp-python[server]"

In [ ]:
# 3. Vérification du Modèle GGUF & Démarrage Persistant de llama-server (Skip si déjà chargé)
import os, subprocess, time, glob, urllib.request
from huggingface_hub import list_repo_files, hf_hub_download

# Test si llama-server tourne déjà sur le port 8080
llama_running = False
try:
    req = urllib.request.urlopen('http://localhost:8080/v1/models', timeout=2)
    if req.getcode() == 200:
        llama_running = True
        print('🔥 llama-server tourne DÉJÀ avec le modèle chargé en RAM ! Téléchargement et rechargement IGNORÉS.')
except Exception:
    pass

if not llama_running:
    repo_id = 'KevinJK51/Qwen3.6-12B-IQ-Ultra-Heretic-Uncensored-Thinking-V2-Hightop-GGUF'
    model_dir = '/kaggle/working/models'
    existing_gguf = glob.glob(f'{model_dir}/*.gguf')
    
    if existing_gguf:
        model_path = existing_gguf[0]
        print(f'🟢 Fichier GGUF déjà présent sur disque : {model_path}')
    else:
        print('🔍 Recherche du fichier GGUF unique...')
        files = list_repo_files(repo_id)
        gguf_files = [f for f in files if f.endswith('.gguf')]
        selected_file = next((f for f in gguf_files if 'iq4' in f.lower() or 'iq3' in f.lower() or 'q4' in f.lower()), gguf_files[0])
        print(f'⬇️ Téléchargement du modèle GGUF : {selected_file}...')
        model_path = hf_hub_download(repo_id=repo_id, filename=selected_file, local_dir=model_dir)
    
    print('🚀 Démarrage de llama-server en arrière-plan...')
    llama_cmd = [
        'python', '-m', 'llama_cpp.server',
        '--model', model_path,
        '--host', '0.0.0.0',
        '--port', '8080',
        '--n_ctx', '32768',
        '--n_threads', '2'
    ]
    subprocess.Popen(llama_cmd)
    time.sleep(8)
    print('🟢 llama-server prêt !')

In [ ]:
# 4. Démarrage de FastAPI & Tunnel HTTPS Cloudflare
import os, sys, subprocess, time, re

backend_dir = '/kaggle/working/projet_osint/backend'
os.chdir(backend_dir)
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

!pip install --no-cache-dir -r requirements.txt
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared || true

# Tuer l'ancien processus FastAPI s'il existe
subprocess.run(['pkill', '-f', 'app.main'])
time.sleep(1)

print('Démarrage du Serveur FastAPI (Port 8000)...')
server_process = subprocess.Popen(['python', '-m', 'app.main'], cwd=backend_dir)
time.sleep(5)

# Tuer l'ancien tunnel s'il existe
subprocess.run(['pkill', '-f', 'cloudflared'])
time.sleep(1)

print('Lancement du Tunnel HTTPS...')
tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(25):
    line = tunnel_process.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('\n======================================================')
            print(f'🚀 INTERFACE EN LIGNE (HOT RELOAD ACTIF) : {match.group(0)}')
            print('======================================================\n')
            break
    time.sleep(1)

In [ ]:
# ⚡ CELLULE HOT-RELOAD : EXÉCUTER CETTE CELLULE APRÈS CHAQUE MODIFICATION SUR GITHUB !
# Temps d'application : 2 secondes (Le modèle LLM reste chargé en mémoire)
import subprocess, time

repo_dir = '/kaggle/working/projet_osint'
backend_dir = '/kaggle/working/projet_osint/backend'

print('🔄 1. Récupération des dernières modifications d\'interface depuis GitHub (Git Pull)...')
subprocess.run(['git', '-C', repo_dir, 'pull', 'origin', 'main'])

print('⚡ 2. Redémarrage instantané du serveur FastAPI (sans toucher au modèle LLM)...')
subprocess.run(['pkill', '-f', 'app.main'])
time.sleep(1)
subprocess.Popen(['python', '-m', 'app.main'], cwd=backend_dir)
time.sleep(3)

print('✅ Interface mise à jour en 2 secondes ! Rechargez votre lien Cloudflare.')